# **Shopee Fake Review Detection**

![Shopee Logo](https://upload.wikimedia.org/wikipedia/commons/0/0e/Shopee_logo.svg)

Shopee merupakan salah satu platform e-commerce terbesar di kawasan Asia Tenggara dan Taiwan. Khususnya di Indonesia, Shopee menjadi salah satu platform yang banyak diminati oleh masyarakat karena berbagai program menarik, salah satunya adalah kebijakan gratis ongkir.

**Lalu, mengapa fake review menjadi permasalahan yang berbahaya?**

Terdapat beberapa alasan mengapa keberadaan ulasan palsu dapat memberikan dampak negatif bagi Shopee sebagai platform e-commerce.

Pertama, berdasarkan penelitian BrightLocal (2017), sebanyak 85% konsumen mempercayai ulasan online dengan tingkat kepercayaan yang setara dengan rekomendasi pribadi. Hal ini menunjukkan bahwa ulasan memiliki peran penting dalam memengaruhi keputusan pembelian konsumen. Oleh karena itu, keberadaan ulasan palsu dapat menyesatkan calon pembeli dan berpotensi merusak kepercayaan konsumen terhadap suatu merek maupun platform.

Kedua, ketika konsumen melakukan pembelian berdasarkan ulasan palsu, tetapi produk yang diterima tidak sesuai dengan ekspektasi yang dibangun oleh ulasan tersebut, konsumen dapat mengalami kekecewaan. Dalam jangka panjang, kondisi ini berpotensi menurunkan tingkat kepercayaan pengguna terhadap Shopee sebagai platform e-commerce serta mengurangi minat konsumen untuk melakukan transaksi kembali. Selain itu, peningkatan jumlah keluhan akibat pengalaman buruk tersebut juga dapat menambah beban kerja pada layanan pelanggan.

Di sisi lain, proses identifikasi dan penghapusan ulasan palsu secara manual membutuhkan waktu dan tenaga yang besar karena jumlah ulasan pada platform e-commerce terus meningkat. Oleh karena itu, diperlukan pendekatan berbasis teknologi AI untuk membantu proses deteksi secara lebih otomatis.

Berdasarkan permasalahan tersebut, use case project ini membangun sebuah algoritma deep learning untuk melakukan klasifikasi ulasan palsu.

**Contoh Fake Review**

![Contoh Fake Review](https://miro.medium.com/v2/resize:fit:640/format:webp/1*dsDSGUbm8tcJIm2vf75Cyw.png)

![Contoh Fake Review](https://i.imgur.com/4uMxr2S.png)





## 1. Install dan import library

Cell ini hanya memanggil library yang dipakai di notebook.

In [1]:
%pip install -q kagglehub
%pip install nltk
%pip install scikit-learn

from collections import Counter
from pathlib import Path

import os
import re
import shutil

import kagglehub
import nltk
import pandas as pd

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from scipy.sparse import hstack, vstack
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Sofia\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Sofia\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Sofia\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Semua file mentah dan hasil preprocessing disimpan dalam satu folder kerja.

In [2]:
NOTEBOOK_ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "USE CASE 12.ipynb").exists()
    ),
    Path.cwd(),
)
WORK_DIR = Path("/content/fake_review_detection") if Path("/content").exists() else NOTEBOOK_ROOT / "fake_review_notebook"
RAW_DIR = WORK_DIR / "raw_data"

WORK_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(WORK_DIR)
PROJECT_ROOT = NOTEBOOK_ROOT

print("Working directory:", WORK_DIR)

Working directory: c:\Users\Sofia\Downloads\digistar\fake_review_notebook


## 2. Data Collection

Pada tahap ini dilakukan proses pengumpulan dataset yang akan digunakan dalam proyek Fake Review Detection.

Dataset dibagi menjadi dua bagian:

1. Dataset utama (`train_review_df`)
   - Diambil dari repository `andrioktavianto/fake-review-shopee` yaitu `train_review_only.csv`
   - Berisi 100 baris ulasan produk Shopee yang telah diberi label secara manual dengan proporsi 51 fake review: 49 original/autentic review.
   - Digunakan sebagai dataset training untuk model supervised baseline. Pendekatan ini dilakukan sebagai proof-of-concept.

1. Dataset Tambahan (`review_df`)
   - Diambil dari repository `andrioktavianto/fake-review-shopee` yaitu `review_shopee.csv`.
   - Berisi kumpulan ulasan produk Shopee yang belum memiliki label.
   - Digunakan untuk memperluas analisis dan berbagai sumber data semi-supervised.


In [3]:
github_base_url = "https://raw.githubusercontent.com/andrioktavianto/fake-review-shopee/master"

github_files = [
    "review_shopee.csv",
    "train_review_only.csv",
]

for filename in github_files:
    url = f"{github_base_url}/{filename}"

    temp_df = pd.read_csv(url)

    temp_df.to_csv(WORK_DIR / filename, index=False)
    temp_df.to_csv(RAW_DIR / filename, index=False)

# Load dua dataset review dari file yang sudah diunduh.
train_review_df = pd.read_csv(
    WORK_DIR / "train_review_only.csv"
)

review_df = pd.read_csv(
    WORK_DIR / "review_shopee.csv"
)

print("Train review dataset:", train_review_df.shape)

display(train_review_df.head())

Train review dataset: (100, 25)


,anonymous,author_shopid,author_username,cat_id,cmtid,comment,count_rating_with_image,count_with_context,ctime,editable,item_id,mtime,orderid,product_title,rating,rating_count0,rating_count1,rating_count2,rating_count3,rating_count4,rating_count5,rating_star,shop_id,userid,fakeornot
0,False,80294228,icanursarah,2902,2327971512,"Hp sampai dengan selamat, berfungsi dengan baik, tiada kendala Alhamdulillah, semoga awet dan mantap jiwaa",922,2182,1588431497,1,2767211185,1588431497,NaN,Nord VPN Lifetime - Bergaransi - Termurah - Bypass Netflix - VPN Netflix - VPN Indihome/Telk,1,7274,89,15,44,88,9088,5,15168157,80295702,fake
1,False,80294228,icanursarah,2902,2325993622,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,922,2182,1588403926,1,2767211185,1588403926,NaN,Nord VPN Lifetime - Bergaransi - Termurah - Bypass Netflix - VPN Netflix - VPN Indihome/Telk,1,7274,89,15,44,88,9088,5,15168157,80295702,fake
2,False,80294228,icanursarah,2902,2325999143,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,922,2182,1588403984,1,2767211185,1588403984,NaN,Nord VPN Lifetime - Bergaransi - Termurah - Bypass Netflix - VPN Netflix - VPN Indihome/Telk,1,7274,89,15,44,88,9088,5,15168157,80295702,fake
3,False,80294228,icanursarah,2902,2326001075,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,922,2182,1588404004,1,2767211185,1588404004,NaN,Nord VPN Lifetime - Bergaransi - Termurah - Bypass Netflix - VPN Netflix - VPN Indihome/Telk,1,7274,89,15,44,88,9088,5,15168157,80295702,fake
4,False,16191982,rrp0811,2678,2361588821,Ini paket pertamaku beli disini.\r\nCepat\r\nTanggap\r\nAkurat\r\nTepat\r\nBagus\r\nOke\r\nSemoga aja tanpa ada masa...,284,680,1589011931,1,4822872755,1589011931,NaN,NETFLIX PREMIUM 1 BULAN 4K ANTI ON,1,1316,2,4,1,44,3584,5,134248812,16193318,fake


In [4]:
print("Additional review dataset:", review_df.shape)

display(review_df.head())

Additional review dataset: (9260, 24)


,anonymous,author_shopid,author_username,cat_id,cmtid,comment,count_rating_with_image,count_with_context,ctime,editable,item_id,mtime,orderid,product_title,rating,rating_count0,rating_count1,rating_count2,rating_count3,rating_count4,rating_count5,rating_star,shop_id,userid
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,False,15568078.0,partianadewi,3268.0,1.940509e+09,work tp proses lama,0.0,5.0,1.578476e+09,0.0,5.502846e+09,1.578476e+09,NaN,[LEGAL] NETFLIX PREMIUM 1 TAHUN FULL GA,1.0,20.0,0.0,0.0,0.0,0.0,14.0,5.0,121336544.0,15569414.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,False,140322961.0,arvie2003,3268.0,1.925999e+09,"Respon cepat, work, dan proses agak lama",0.0,5.0,1.578085e+09,0.0,5.502846e+09,1.578085e+09,NaN,[LEGAL] NETFLIX PREMIUM 1 TAHUN FULL GA,1.0,20.0,0.0,0.0,0.0,0.0,14.0,5.0,121336544.0,140324785.0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
from pathlib import Path
print("cwd:", Path.cwd())
print("WORK_DIR:", WORK_DIR)
print("RAW_DIR:", RAW_DIR)
print("product_path:", product_path)
print("Exists at WORK_DIR:", (WORK_DIR / "dataset_panci_wajan.csv").exists())
print("Files in WORK_DIR:", [p.name for p in WORK_DIR.glob("*")])
# list CSVs under product_path
print("CSVs under product_path:", [p.name for p in Path(product_path).rglob("*.csv")])

cwd: c:\Users\Sofia\Downloads\digistar\fake_review_notebook
WORK_DIR: c:\Users\Sofia\Downloads\digistar\fake_review_notebook
RAW_DIR: c:\Users\Sofia\Downloads\digistar\fake_review_notebook\raw_data
product_path: c:\Users\Sofia\Downloads\digistar
Exists at WORK_DIR: True
Files in WORK_DIR: ['dataset_panci_wajan.csv', 'raw_data', 'review_shopee.csv', 'train_review_only.csv']
CSVs under product_path: ['dataset_panci_wajan.csv', 'review_shopee.csv', 'shopee_ulasan_label7.csv', 'train_review_only.csv', 'dataset_panci_wajan.csv', 'review_shopee.csv', 'train_review_only.csv', 'dataset_panci_wajan.csv', 'review_shopee.csv', 'train_review_only.csv', 'shopee_fake_review_modeling_dataset.csv', 'fr_FR.csv', 'breast_cancer.csv', 'iris.csv', 'linnerud_exercise.csv', 'linnerud_physiological.csv', 'wine_data.csv', 'GLB.Ts+dSST.csv', 'umath-validation-set-arccos.csv', 'umath-validation-set-arccosh.csv', 'umath-validation-set-arcsin.csv', 'umath-validation-set-arcsinh.csv', 'umath-validation-set-arctan.

## 3. Preprocessing dan Feature Engineering untuk train_review_df

Tahap data cleaning dilakukan untuk mempersiapkan dataset sebelum masuk ke tahap pemodelan machine learning.

Pada tahap ini dilakukan:
1. Penanganan missing value pada data review.
2. Pembersihan teks (*text cleaning*).
3. Normalisasi teks.
4. Lowercase conversion.
5. Penghapusan stopword.

Data duplikat tidak dihapus karena pola review yang berulang dapat menjadi informasi penting dalam proses deteksi fake review.

In [7]:
train_review_df.describe()

,author_shopid,cat_id,cmtid,count_rating_with_image,count_with_context,ctime,editable,item_id,mtime,orderid,rating,rating_count0,rating_count1,rating_count2,rating_count3,rating_count4,rating_count5,rating_star,shop_id,userid
count,1.000000e+02,100.000000,1.000000e+02,100.000000,100.000000,1.000000e+02,100.000000,1.000000e+02,1.000000e+02,0.0,100.00000,100.00000,100.000000,100.00000,100.000000,100.000000,100.000000,100.000000,1.000000e+02,1.000000e+02
mean,9.165345e+07,2678.650000,2.313313e+09,715.080000,1259.940000,1.587748e+09,0.880000,3.835310e+09,1.587779e+09,NaN,0.83000,5463.27000,132.670000,15.98000,31.050000,87.550000,6961.970000,4.640000,8.219773e+07,9.165527e+07
std,7.629192e+07,577.913442,1.778786e+08,861.705083,1372.139941,4.372252e+06,0.477367,2.092785e+09,4.334472e+06,NaN,0.55149,6198.81407,224.825681,24.49159,42.989399,116.007347,8529.522252,1.123846,5.879091e+07,7.629268e+07
min,3.614449e+06,42.000000,1.414417e+09,0.000000,5.000000,1.564130e+09,0.000000,8.451010e+08,1.564130e+09,NaN,-1.00000,17.00000,0.000000,0.00000,0.000000,0.000000,14.000000,1.000000,1.912337e+06,3.615821e+06
25%,2.942194e+07,2160.000000,2.319222e+09,42.750000,56.000000,1.588259e+09,1.000000,2.354260e+09,1.588259e+09,NaN,1.00000,254.50000,1.750000,0.00000,0.750000,2.750000,148.000000,5.000000,1.978175e+07,2.942333e+07
50%,8.029423e+07,2580.000000,2.338198e+09,284.000000,680.000000,1.588655e+09,1.000000,2.767211e+09,1.588655e+09,NaN,1.00000,1551.00000,10.000000,4.00000,4.500000,20.000000,2449.500000,5.000000,7.401964e+07,8.029570e+07
75%,1.188133e+08,2902.000000,2.422795e+09,922.000000,2182.000000,1.589964e+09,1.000000,5.505998e+09,1.589964e+09,NaN,1.00000,10259.00000,89.000000,15.00000,44.000000,96.000000,9088.000000,5.000000,1.561384e+08,1.188151e+08
max,2.676015e+08,3960.000000,2.463434e+09,2412.000000,3626.000000,1.590795e+09,2.000000,7.832626e+09,1.590795e+09,NaN,1.00000,16647.00000,612.000000,67.00000,113.000000,309.000000,23513.000000,5.000000,1.720565e+08,2.676048e+08


In [8]:
# Check null values
train_review_df.isna().sum()

anonymous                    0
author_shopid                0
author_username              0
cat_id                       0
cmtid                        0
comment                      1
count_rating_with_image      0
count_with_context           0
ctime                        0
editable                     0
item_id                      0
mtime                        0
orderid                    100
product_title                0
rating                       0
rating_count0                0
rating_count1                0
rating_count2                0
rating_count3                0
rating_count4                0
rating_count5                0
rating_star                  0
shop_id                      0
userid                       0
fakeornot                    0
dtype: int64

In [9]:
train_review_df = train_review_df.dropna(subset=["comment"])
train_review_df.isna().sum()

anonymous                   0
author_shopid               0
author_username             0
cat_id                      0
cmtid                       0
comment                     0
count_rating_with_image     0
count_with_context          0
ctime                       0
editable                    0
item_id                     0
mtime                       0
orderid                    99
product_title               0
rating                      0
rating_count0               0
rating_count1               0
rating_count2               0
rating_count3               0
rating_count4               0
rating_count5               0
rating_star                 0
shop_id                     0
userid                      0
fakeornot                   0
dtype: int64

In [10]:
train_review_df['fakeornot'].value_counts()

fakeornot
fake        50
original    49
Name: count, dtype: int64

In [11]:
train_review_df['product_title'].value_counts()

product_title
NETFLIX PREMIUM LIFETIME UHD GA                                                                       17
Nord VPN Lifetime - Bergaransi - Termurah - Bypass Netflix - VPN Netflix - VPN Indihome/Telk          12
NETFLIX PREMIUM PRIVATE UHD GARANSI FULL ATAU SH                                                       6
NETFLIX PREMIUM ULTRA HD 4K PRIVATE ACCOUNT 1                                                          6
NETFLIX PREMIUM LIFETIME GARANS                                                                        5
NETFLIX PREMIUM 1 BULAN 4K ANTI ON                                                                     4
[TRUSTED]NETFLIX PREMIUM, 4K 4 Devices - Bukan Shared, Resmi dan Legal                                 4
Netflix Premium 1 Bulan Anti B                                                                         3
[TERLARIS] Netflix Premium Plan Akun Pribadi & Sharing (1 Bulan) GARANSI + NOR                         3
Netflix Premium Anti On Hold 30          

In [12]:
# Preprocessing
# Tokenization

train_review_df['comment_token'] = train_review_df['comment'].apply(
    lambda x: word_tokenize(str(x))
)

train_review_df[['orderid', 'comment', 'comment_token']].head()

,orderid,comment,comment_token
0,NaN,"Hp sampai dengan selamat, berfungsi dengan baik, tiada kendala Alhamdulillah, semoga awet dan mantap jiwaa","[Hp, sampai, dengan, selamat, ,, berfungsi, dengan, baik, ,, tiada, kendala, Alhamdulillah, ,, semoga, awet, dan, ma..."
1,NaN,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,"[Mantabbagusmantabbagus, yuhuuy, Recommended, seller, yuhuuuyyy]"
2,NaN,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,"[Mantabbagusmantabbagus, yuhuuy, Recommended, seller, yuhuuuyyy]"
3,NaN,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,"[Mantabbagusmantabbagus, yuhuuy, Recommended, seller, yuhuuuyyy]"
4,NaN,Ini paket pertamaku beli disini.\r\nCepat\r\nTanggap\r\nAkurat\r\nTepat\r\nBagus\r\nOke\r\nSemoga aja tanpa ada masa...,"[Ini, paket, pertamaku, beli, disini, ., Cepat, Tanggap, Akurat, Tepat, Bagus, Oke, Semoga, aja, tanpa, ada, masalah..."


In [13]:
# Preprocessing
# Lowercase Conversion

train_review_df['comment_lower'] = train_review_df['comment_token'].apply(
    lambda x: [word.lower() for word in x]
)

train_review_df[['orderid', 'comment', 'comment_token', 'comment_lower']].head()

,orderid,comment,comment_token,comment_lower
0,NaN,"Hp sampai dengan selamat, berfungsi dengan baik, tiada kendala Alhamdulillah, semoga awet dan mantap jiwaa","[Hp, sampai, dengan, selamat, ,, berfungsi, dengan, baik, ,, tiada, kendala, Alhamdulillah, ,, semoga, awet, dan, ma...","[hp, sampai, dengan, selamat, ,, berfungsi, dengan, baik, ,, tiada, kendala, alhamdulillah, ,, semoga, awet, dan, ma..."
1,NaN,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,"[Mantabbagusmantabbagus, yuhuuy, Recommended, seller, yuhuuuyyy]","[mantabbagusmantabbagus, yuhuuy, recommended, seller, yuhuuuyyy]"
2,NaN,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,"[Mantabbagusmantabbagus, yuhuuy, Recommended, seller, yuhuuuyyy]","[mantabbagusmantabbagus, yuhuuy, recommended, seller, yuhuuuyyy]"
3,NaN,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,"[Mantabbagusmantabbagus, yuhuuy, Recommended, seller, yuhuuuyyy]","[mantabbagusmantabbagus, yuhuuy, recommended, seller, yuhuuuyyy]"
4,NaN,Ini paket pertamaku beli disini.\r\nCepat\r\nTanggap\r\nAkurat\r\nTepat\r\nBagus\r\nOke\r\nSemoga aja tanpa ada masa...,"[Ini, paket, pertamaku, beli, disini, ., Cepat, Tanggap, Akurat, Tepat, Bagus, Oke, Semoga, aja, tanpa, ada, masalah...","[ini, paket, pertamaku, beli, disini, ., cepat, tanggap, akurat, tepat, bagus, oke, semoga, aja, tanpa, ada, masalah..."


In [14]:
# Preprocessing
# Remove Punctuations

train_review_df['comment_punct'] = train_review_df['comment_lower'].apply(
    lambda x: [word for word in x if re.search(r'\w', word)]
)

# Tampilkan hasil
train_review_df[['orderid', 'comment', 'comment_token', 'comment_lower', 'comment_punct']].head()

,orderid,comment,comment_token,comment_lower,comment_punct
0,NaN,"Hp sampai dengan selamat, berfungsi dengan baik, tiada kendala Alhamdulillah, semoga awet dan mantap jiwaa","[Hp, sampai, dengan, selamat, ,, berfungsi, dengan, baik, ,, tiada, kendala, Alhamdulillah, ,, semoga, awet, dan, ma...","[hp, sampai, dengan, selamat, ,, berfungsi, dengan, baik, ,, tiada, kendala, alhamdulillah, ,, semoga, awet, dan, ma...","[hp, sampai, dengan, selamat, berfungsi, dengan, baik, tiada, kendala, alhamdulillah, semoga, awet, dan, mantap, jiwaa]"
1,NaN,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,"[Mantabbagusmantabbagus, yuhuuy, Recommended, seller, yuhuuuyyy]","[mantabbagusmantabbagus, yuhuuy, recommended, seller, yuhuuuyyy]","[mantabbagusmantabbagus, yuhuuy, recommended, seller, yuhuuuyyy]"
2,NaN,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,"[Mantabbagusmantabbagus, yuhuuy, Recommended, seller, yuhuuuyyy]","[mantabbagusmantabbagus, yuhuuy, recommended, seller, yuhuuuyyy]","[mantabbagusmantabbagus, yuhuuy, recommended, seller, yuhuuuyyy]"
3,NaN,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,"[Mantabbagusmantabbagus, yuhuuy, Recommended, seller, yuhuuuyyy]","[mantabbagusmantabbagus, yuhuuy, recommended, seller, yuhuuuyyy]","[mantabbagusmantabbagus, yuhuuy, recommended, seller, yuhuuuyyy]"
4,NaN,Ini paket pertamaku beli disini.\r\nCepat\r\nTanggap\r\nAkurat\r\nTepat\r\nBagus\r\nOke\r\nSemoga aja tanpa ada masa...,"[Ini, paket, pertamaku, beli, disini, ., Cepat, Tanggap, Akurat, Tepat, Bagus, Oke, Semoga, aja, tanpa, ada, masalah...","[ini, paket, pertamaku, beli, disini, ., cepat, tanggap, akurat, tepat, bagus, oke, semoga, aja, tanpa, ada, masalah...","[ini, paket, pertamaku, beli, disini, cepat, tanggap, akurat, tepat, bagus, oke, semoga, aja, tanpa, ada, masalah, d..."


In [15]:
# Preprocessing
# Stopword Removal

stop_words = set(stopwords.words('indonesian'))

train_review_df['comment_stopword'] = train_review_df['comment_punct'].apply(
    lambda x: [word for word in x if word not in stop_words]
)

# Tampilkan hasil
train_review_df[['orderid', 'comment', 'comment_token', 'comment_lower', 'comment_punct', 'comment_stopword']].head()

,orderid,comment,comment_token,comment_lower,comment_punct,comment_stopword
0,NaN,"Hp sampai dengan selamat, berfungsi dengan baik, tiada kendala Alhamdulillah, semoga awet dan mantap jiwaa","[Hp, sampai, dengan, selamat, ,, berfungsi, dengan, baik, ,, tiada, kendala, Alhamdulillah, ,, semoga, awet, dan, ma...","[hp, sampai, dengan, selamat, ,, berfungsi, dengan, baik, ,, tiada, kendala, alhamdulillah, ,, semoga, awet, dan, ma...","[hp, sampai, dengan, selamat, berfungsi, dengan, baik, tiada, kendala, alhamdulillah, semoga, awet, dan, mantap, jiwaa]","[hp, selamat, berfungsi, tiada, kendala, alhamdulillah, semoga, awet, mantap, jiwaa]"
1,NaN,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,"[Mantabbagusmantabbagus, yuhuuy, Recommended, seller, yuhuuuyyy]","[mantabbagusmantabbagus, yuhuuy, recommended, seller, yuhuuuyyy]","[mantabbagusmantabbagus, yuhuuy, recommended, seller, yuhuuuyyy]","[mantabbagusmantabbagus, yuhuuy, recommended, seller, yuhuuuyyy]"
2,NaN,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,"[Mantabbagusmantabbagus, yuhuuy, Recommended, seller, yuhuuuyyy]","[mantabbagusmantabbagus, yuhuuy, recommended, seller, yuhuuuyyy]","[mantabbagusmantabbagus, yuhuuy, recommended, seller, yuhuuuyyy]","[mantabbagusmantabbagus, yuhuuy, recommended, seller, yuhuuuyyy]"
3,NaN,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,"[Mantabbagusmantabbagus, yuhuuy, Recommended, seller, yuhuuuyyy]","[mantabbagusmantabbagus, yuhuuy, recommended, seller, yuhuuuyyy]","[mantabbagusmantabbagus, yuhuuy, recommended, seller, yuhuuuyyy]","[mantabbagusmantabbagus, yuhuuy, recommended, seller, yuhuuuyyy]"
4,NaN,Ini paket pertamaku beli disini.\r\nCepat\r\nTanggap\r\nAkurat\r\nTepat\r\nBagus\r\nOke\r\nSemoga aja tanpa ada masa...,"[Ini, paket, pertamaku, beli, disini, ., Cepat, Tanggap, Akurat, Tepat, Bagus, Oke, Semoga, aja, tanpa, ada, masalah...","[ini, paket, pertamaku, beli, disini, ., cepat, tanggap, akurat, tepat, bagus, oke, semoga, aja, tanpa, ada, masalah...","[ini, paket, pertamaku, beli, disini, cepat, tanggap, akurat, tepat, bagus, oke, semoga, aja, tanpa, ada, masalah, d...","[paket, pertamaku, beli, cepat, tanggap, akurat, bagus, oke, semoga, aja, ditengah, jalan, jsjsjjdjdhshwjshzvzvvzhaj..."


In [16]:
# Gabungkan token menjadi teks kembali
train_review_df['comment_clean'] = train_review_df['comment_punct'].apply(
    lambda x: ' '.join(x)
)

# Hapus kolom preprocessing sementara
cols_to_drop = [
    'comment_token',
    'comment_lower',
    'comment_punct',
    'comment_stopword'
]

train_review_df = train_review_df.drop(
    columns=[col for col in cols_to_drop if col in train_review_df.columns]
)

# Tampilkan dataframe
train_review_df

,anonymous,author_shopid,author_username,cat_id,cmtid,comment,count_rating_with_image,count_with_context,ctime,editable,item_id,mtime,orderid,product_title,rating,rating_count0,rating_count1,rating_count2,rating_count3,rating_count4,rating_count5,rating_star,shop_id,userid,fakeornot,comment_clean
0,False,80294228,icanursarah,2902,2327971512,"Hp sampai dengan selamat, berfungsi dengan baik, tiada kendala Alhamdulillah, semoga awet dan mantap jiwaa",922,2182,1588431497,1,2767211185,1588431497,NaN,Nord VPN Lifetime - Bergaransi - Termurah - Bypass Netflix - VPN Netflix - VPN Indihome/Telk,1,7274,89,15,44,88,9088,5,15168157,80295702,fake,hp sampai dengan selamat berfungsi dengan baik tiada kendala alhamdulillah semoga awet dan mantap jiwaa
1,False,80294228,icanursarah,2902,2325993622,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,922,2182,1588403926,1,2767211185,1588403926,NaN,Nord VPN Lifetime - Bergaransi - Termurah - Bypass Netflix - VPN Netflix - VPN Indihome/Telk,1,7274,89,15,44,88,9088,5,15168157,80295702,fake,mantabbagusmantabbagus yuhuuy recommended seller yuhuuuyyy
2,False,80294228,icanursarah,2902,2325999143,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,922,2182,1588403984,1,2767211185,1588403984,NaN,Nord VPN Lifetime - Bergaransi - Termurah - Bypass Netflix - VPN Netflix - VPN Indihome/Telk,1,7274,89,15,44,88,9088,5,15168157,80295702,fake,mantabbagusmantabbagus yuhuuy recommended seller yuhuuuyyy
3,False,80294228,icanursarah,2902,2326001075,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,922,2182,1588404004,1,2767211185,1588404004,NaN,Nord VPN Lifetime - Bergaransi - Termurah - Bypass Netflix - VPN Netflix - VPN Indihome/Telk,1,7274,89,15,44,88,9088,5,15168157,80295702,fake,mantabbagusmantabbagus yuhuuy recommended seller yuhuuuyyy
4,False,16191982,rrp0811,2678,2361588821,Ini paket pertamaku beli disini.\r\nCepat\r\nTanggap\r\nAkurat\r\nTepat\r\nBagus\r\nOke\r\nSemoga aja tanpa ada masa...,284,680,1589011931,1,4822872755,1589011931,NaN,NETFLIX PREMIUM 1 BULAN 4K ANTI ON,1,1316,2,4,1,44,3584,5,134248812,16193318,fake,ini paket pertamaku beli disini cepat tanggap akurat tepat bagus oke semoga aja tanpa ada masalah ditengah jalan jsj...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,False,36425398,achiesaurus,3176,1494231889,Bulan 1 lancar aksesnya. Semoga bulan selanjutnya lantjar djaya juga. 👍,30,85,1566739325,0,896272444,1566739325,NaN,[TRUSTED]NETFLIX PAKET 1 TAHUN PREMIUM 100% aman dan,1,247,3,0,0,4,293,5,52886923,36426782,original,bulan 1 lancar aksesnya semoga bulan selanjutnya lantjar djaya juga
96,False,149792766,nizartegar,3530,2463434315,Pemesanan kedua. Bulan pertama aman lancar tanpa ada kendala sama sekali. Seller fast respon dan baik. Pertahankan k...,57,214,1590794610,1,2373540050,1590794610,NaN,(1BULAN = 29K) NETFLIX PREMIUM ULTRA HD 4K ANTI ONHOLD 1 BULAN RESMI BERGARANSI TERMURAH,1,829,3,0,0,28,770,5,59508259,149794590,original,pemesanan kedua bulan pertama aman lancar tanpa ada kendala sama sekali seller fast respon dan baik pertahankan kual...
97,False,153074375,raihan.mf,3530,2391633492,"Adminnya fast resp, dilayani dengan ramah dan ga ngancem blok klo ga di review WKWKWKWK, definitely bakal perpanjang...",57,214,1589469468,1,2373540050,1589469468,NaN,(1BULAN = 29K) NETFLIX PREMIUM ULTRA HD 4K ANTI ONHOLD 1 BULAN RESMI BERGARANSI TERMURAH,1,829,3,0,0,28,770,5,59508259,153076248,original,adminnya fast resp dilayani dengan ramah dan ga ngancem blok klo ga di review wkwkwkwk definitely bakal perpanjang b...
98,False,199142393,ayasehamasaki,3960,2398772567,Makasiiii banyak! Udah beberapa hari ini nonton lancar2 aja. Proses mudah dan adminnya helpful sekali.🙂🙂🙂,47,56,1589591903,1,7229496558,1589591903,NaN,Netflix Premium Anti On Hold 30,1,257,0,0,2,8,148,5,136191258,199145493,original,makasiiii banyak udah beberapa hari ini nonton lancar2 aja proses mudah dan adminnya helpful sekali.🙂🙂🙂


In [17]:
# Feature Engineering - TF-IDF

tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(
    train_review_df['comment_clean']
)

tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1451 stored elements and shape (99, 630)>

In [18]:
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf.get_feature_names_out()
)

tfidf_df.head()

,10,100,11,12,1minggu,2hari,30,32,64,660,abis,account,ada,admin,adminnya,agak,agan,ahaaayyy,aja,akan,akhir,akhirnya,aksesnya,aksjcv,akun,akun2,akunnya,akurat,alhamdulillah,allah,always,aman,amat,amazing,aminnnn,ampas,ampe,an,and,annual,anti,anw,apa,apa2,apapun,apk,app,asyikkkkkkkkk,atasi,atau,...,toko,tokonya,tonton,tp,transaksinya,trial,trus,trusted,tv,uang,uda,udah,udahlah,udh,ujung2nya,ulang,uninstall,untuk,untung,upgrade,us,user,usernya,utk,uwhqn,very,voucher,vpn,wa,wajib,waktu,walaupun,web,wkwkwkwk,work,worked,working,wow,ya,yaa,yaaaa,yahuuud,yang,yayyy,yg,yuhuuu,yuhuuuy,yuhuuuyyy,yuhuuy,zonkk
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.265116,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.424525,0.558792,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.424525,0.558792,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.424525,0.558792,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.166514,0.0,0.0,0.0,0.0,0.0,0.21408,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.252992,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0


In [19]:
# Feature Engineering
train_review_df['review_length'] = train_review_df['comment_clean'].apply(
    lambda x: len(x.split())
)

# Ambil 5 fake dan 5 original secara random
fake_sample = train_review_df[train_review_df['fakeornot'] == 'fake'].sample(
    5, random_state=42
)
original_sample = train_review_df[train_review_df['fakeornot'] == 'original'].sample(
    5, random_state=42
)

sample_review = pd.concat([fake_sample, original_sample])
sample_review[['orderid', 'comment', 'comment_clean', 'fakeornot', 'review_length']]

,orderid,comment,comment_clean,fakeornot,review_length
13,NaN,bagus mantap keren wow gila banget amazing baik trusted cepat 100 persen bagus mantap keren wow gila banget amazing ...,bagus mantap keren wow gila banget amazing baik trusted cepat 100 persen bagus mantap keren wow gila banget amazing ...,fake,50
39,NaN,Recommended seller yuhuuuyyy\r\nMantab manfaat mantab yayyy,recommended seller yuhuuuyyy mantab manfaat mantab yayyy,fake,7
30,NaN,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,mantabbagusmantabbagus yuhuuy recommended seller yuhuuuyyy,fake,5
45,NaN,"sukaa banget sama pelayanannyaa,sellernyaa juga ramaah bangett",sukaa banget sama pelayanannyaa sellernyaa juga ramaah bangett,fake,8
17,NaN,"Produk nya berkualitas banget\r\nSeller responsif ehehe\r\nFastresp juga termasuknya\r\nTrusted kok ehehe,udah banya...",produk nya berkualitas banget seller responsif ehehe fastresp juga termasuknya trusted kok ehehe udah banyak yang pe...,fake,19
64,NaN,Alhamdulillah cepat sekali..malah saya yg slowrespon karena sibuk hehe\r\nRespon penjual baik\r\nInsyaAllah langgana...,alhamdulillah cepat sekali malah saya yg slowrespon karena sibuk hehe respon penjual baik insyaallah langganan deh,original,16
96,NaN,Pemesanan kedua. Bulan pertama aman lancar tanpa ada kendala sama sekali. Seller fast respon dan baik. Pertahankan k...,pemesanan kedua bulan pertama aman lancar tanpa ada kendala sama sekali seller fast respon dan baik pertahankan kual...,original,25
98,NaN,Makasiiii banyak! Udah beberapa hari ini nonton lancar2 aja. Proses mudah dan adminnya helpful sekali.🙂🙂🙂,makasiiii banyak udah beberapa hari ini nonton lancar2 aja proses mudah dan adminnya helpful sekali.🙂🙂🙂,original,15
95,NaN,Bulan 1 lancar aksesnya. Semoga bulan selanjutnya lantjar djaya juga. 👍,bulan 1 lancar aksesnya semoga bulan selanjutnya lantjar djaya juga,original,10
68,NaN,"Langsung bisa digunakan, bisa untuk netflix.. Terimakasih banyak",langsung bisa digunakan bisa untuk netflix terimakasih banyak,original,8


In [20]:
# Feature Engineering
# Positive Word Count and Ratio

positive_words = {
    'bagus', 'mantap', 'baik', 'recommended', 'rekomendasi',
    'terbaik', 'murah', 'cepat', 'puas', 'suka', 'keren',
    'oke', 'ok', 'aman', 'rapi', 'sesuai', 'worth', 'top'
}

train_review_df['positive_word_count'] = train_review_df['comment_clean'].apply(
    lambda x: sum(1 for word in x.split() if word in positive_words)
)

train_review_df['positive_word_ratio'] = train_review_df.apply(
    lambda row: row['positive_word_count'] / row['review_length']
    if row['review_length'] > 0 else 0,
    axis=1
)

# Ambil sample
fake_sample = train_review_df[train_review_df['fakeornot'] == 'fake'].sample(
    5, random_state=42
)

original_sample = train_review_df[train_review_df['fakeornot'] == 'original'].sample(
    5, random_state=42
)

sample_review = pd.concat([fake_sample, original_sample])

sample_review[
    [
        'comment',
        'comment_clean',
        'fakeornot',
        'positive_word_count',
        'positive_word_ratio'
    ]
]

,comment,comment_clean,fakeornot,positive_word_count,positive_word_ratio
13,bagus mantap keren wow gila banget amazing baik trusted cepat 100 persen bagus mantap keren wow gila banget amazing ...,bagus mantap keren wow gila banget amazing baik trusted cepat 100 persen bagus mantap keren wow gila banget amazing ...,fake,21,0.420000
39,Recommended seller yuhuuuyyy\r\nMantab manfaat mantab yayyy,recommended seller yuhuuuyyy mantab manfaat mantab yayyy,fake,1,0.142857
30,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,mantabbagusmantabbagus yuhuuy recommended seller yuhuuuyyy,fake,1,0.200000
45,"sukaa banget sama pelayanannyaa,sellernyaa juga ramaah bangett",sukaa banget sama pelayanannyaa sellernyaa juga ramaah bangett,fake,0,0.000000
17,"Produk nya berkualitas banget\r\nSeller responsif ehehe\r\nFastresp juga termasuknya\r\nTrusted kok ehehe,udah banya...",produk nya berkualitas banget seller responsif ehehe fastresp juga termasuknya trusted kok ehehe udah banyak yang pe...,fake,0,0.000000
64,Alhamdulillah cepat sekali..malah saya yg slowrespon karena sibuk hehe\r\nRespon penjual baik\r\nInsyaAllah langgana...,alhamdulillah cepat sekali malah saya yg slowrespon karena sibuk hehe respon penjual baik insyaallah langganan deh,original,2,0.125000
96,Pemesanan kedua. Bulan pertama aman lancar tanpa ada kendala sama sekali. Seller fast respon dan baik. Pertahankan k...,pemesanan kedua bulan pertama aman lancar tanpa ada kendala sama sekali seller fast respon dan baik pertahankan kual...,original,2,0.080000
98,Makasiiii banyak! Udah beberapa hari ini nonton lancar2 aja. Proses mudah dan adminnya helpful sekali.🙂🙂🙂,makasiiii banyak udah beberapa hari ini nonton lancar2 aja proses mudah dan adminnya helpful sekali.🙂🙂🙂,original,0,0.000000
95,Bulan 1 lancar aksesnya. Semoga bulan selanjutnya lantjar djaya juga. 👍,bulan 1 lancar aksesnya semoga bulan selanjutnya lantjar djaya juga,original,0,0.000000
68,"Langsung bisa digunakan, bisa untuk netflix.. Terimakasih banyak",langsung bisa digunakan bisa untuk netflix terimakasih banyak,original,0,0.000000


In [21]:
# Feature Engineering
# Repeated Word Count and Ratio

train_review_df['repeated_word_count'] = train_review_df['comment_clean'].apply(
    lambda x: sum(
        count - 1 
        for count in Counter(x.split()).values()
        if count > 1
    )
)

train_review_df['repetition_score'] = train_review_df.apply(
    lambda row: row['repeated_word_count'] / row['review_length']
    if row['review_length'] > 0 else 0,
    axis=1
)

# Sample
fake_sample = train_review_df[train_review_df['fakeornot'] == 'fake'].sample(
    5, random_state=42
)

original_sample = train_review_df[train_review_df['fakeornot'] == 'original'].sample(
    5, random_state=42
)

sample_review = pd.concat([fake_sample, original_sample])

sample_review[
    [
        'comment',
        'fakeornot',
        'repeated_word_count',
        'repetition_score'
    ]
]

,comment,fakeornot,repeated_word_count,repetition_score
13,bagus mantap keren wow gila banget amazing baik trusted cepat 100 persen bagus mantap keren wow gila banget amazing ...,fake,36,0.720000
39,Recommended seller yuhuuuyyy\r\nMantab manfaat mantab yayyy,fake,1,0.142857
30,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,fake,0,0.000000
45,"sukaa banget sama pelayanannyaa,sellernyaa juga ramaah bangett",fake,0,0.000000
17,"Produk nya berkualitas banget\r\nSeller responsif ehehe\r\nFastresp juga termasuknya\r\nTrusted kok ehehe,udah banya...",fake,2,0.105263
64,Alhamdulillah cepat sekali..malah saya yg slowrespon karena sibuk hehe\r\nRespon penjual baik\r\nInsyaAllah langgana...,original,0,0.000000
96,Pemesanan kedua. Bulan pertama aman lancar tanpa ada kendala sama sekali. Seller fast respon dan baik. Pertahankan k...,original,0,0.000000
98,Makasiiii banyak! Udah beberapa hari ini nonton lancar2 aja. Proses mudah dan adminnya helpful sekali.🙂🙂🙂,original,0,0.000000
95,Bulan 1 lancar aksesnya. Semoga bulan selanjutnya lantjar djaya juga. 👍,original,1,0.100000
68,"Langsung bisa digunakan, bisa untuk netflix.. Terimakasih banyak",original,1,0.125000


In [22]:
# Feature Engineering
# Frequency of Reviews by User

train_review_df['ctime_datetime'] = pd.to_datetime(
    train_review_df['ctime'],
    unit='s'
)

train_review_df['review_date'] = (
    train_review_df['ctime_datetime'].dt.date
)

# Jumlah review user
train_review_df['user_review_count'] = (
    train_review_df.groupby('userid')['userid']
    .transform('count')
)

# Jumlah review user per hari
train_review_df['user_review_per_day'] = (
    train_review_df
    .groupby(['userid', 'review_date'])['userid']
    .transform('count')
)

# Sample
fake_sample = train_review_df[train_review_df['fakeornot'] == 'fake'].sample(
    5, random_state=42
)

original_sample = train_review_df[train_review_df['fakeornot'] == 'original'].sample(
    5, random_state=42
)

sample_review = pd.concat([fake_sample, original_sample])

sample_review[
    [
        'userid',
        'comment',
        'fakeornot',
        'user_review_count',
        'user_review_per_day'
    ]
]

,userid,comment,fakeornot,user_review_count,user_review_per_day
13,48258995,bagus mantap keren wow gila banget amazing baik trusted cepat 100 persen bagus mantap keren wow gila banget amazing ...,fake,2,1
39,80295702,Recommended seller yuhuuuyyy\r\nMantab manfaat mantab yayyy,fake,23,3
30,80295702,Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy,fake,23,9
45,94783477,"sukaa banget sama pelayanannyaa,sellernyaa juga ramaah bangett",fake,2,2
17,22987580,"Produk nya berkualitas banget\r\nSeller responsif ehehe\r\nFastresp juga termasuknya\r\nTrusted kok ehehe,udah banya...",fake,4,4
64,87081505,Alhamdulillah cepat sekali..malah saya yg slowrespon karena sibuk hehe\r\nRespon penjual baik\r\nInsyaAllah langgana...,original,1,1
96,149794590,Pemesanan kedua. Bulan pertama aman lancar tanpa ada kendala sama sekali. Seller fast respon dan baik. Pertahankan k...,original,1,1
98,199145493,Makasiiii banyak! Udah beberapa hari ini nonton lancar2 aja. Proses mudah dan adminnya helpful sekali.🙂🙂🙂,original,1,1
95,36426782,Bulan 1 lancar aksesnya. Semoga bulan selanjutnya lantjar djaya juga. 👍,original,1,1
68,8614600,"Langsung bisa digunakan, bisa untuk netflix.. Terimakasih banyak",original,1,1


In [23]:
# Feature Engineering
# Cosine Similarity of Reviews

# max_cosine_similarity tidak dihitung di seluruh train_review_df sebelum split,
# karena test data bisa ikut memengaruhi nilai similarity.
# Fitur ini dihitung ulang setelah train-test split pada bagian AI Modeling.

print("max_cosine_similarity dihitung setelah split agar tidak terjadi leakage.")

max_cosine_similarity dihitung setelah split agar tidak terjadi leakage.


## 4. Baseline AI Modeling 

In [24]:
# Encode label dan split train-test tanpa duplicate leakage

train_review_df['label'] = train_review_df['fakeornot'].map({
    'original': 0,
    'fake': 1
})

train_review_model_df = train_review_df.dropna(
    subset=['comment_clean', 'label']
).copy()

comment_group_df = (
    train_review_model_df
    .groupby('comment_clean', as_index=False)
    .agg(
        label=('label', lambda x: x.mode().iat[0]),
        rows=('label', 'size')
    )
)

train_comment_groups, test_comment_groups = train_test_split(
    comment_group_df['comment_clean'],
    test_size=0.3,
    random_state=42,
    stratify=comment_group_df['label']
)

train_review_train_df = train_review_model_df[
    train_review_model_df['comment_clean'].isin(train_comment_groups)
].copy()

train_review_test_df = train_review_model_df[
    train_review_model_df['comment_clean'].isin(test_comment_groups)
].copy()

label_map = {
    0: 'original',
    1: 'fake'
}

train_texts = set(train_review_train_df['comment_clean'])
leaked_test = train_review_test_df[
    train_review_test_df['comment_clean'].isin(train_texts)
]

split_summary_df = pd.DataFrame({
    'data': ['train_review_train_df', 'train_review_test_df', 'leaked_test'],
    'rows': [len(train_review_train_df), len(train_review_test_df), len(leaked_test)],
    'label_fake': [
        (train_review_train_df['fakeornot'] == 'fake').sum(),
        (train_review_test_df['fakeornot'] == 'fake').sum(),
        (leaked_test['fakeornot'] == 'fake').sum()
    ],
    'label_original': [
        (train_review_train_df['fakeornot'] == 'original').sum(),
        (train_review_test_df['fakeornot'] == 'original').sum(),
        (leaked_test['fakeornot'] == 'original').sum()
    ]
})

display(split_summary_df)
leaked_test[['comment', 'comment_clean', 'fakeornot']]

,data,rows,label_fake,label_original
0,train_review_train_df,68,34,34
1,train_review_test_df,31,16,15
2,leaked_test,0,0,0


,comment,comment_clean,fakeornot


In [25]:
# TF-IDF dan fitur numerik setelah split

model_feature_cols = [
    'review_length',
    'repetition_score',
    'positive_word_ratio',
    'user_review_count',
    'user_review_per_day',
    'max_cosine_similarity'
]

cosine_tfidf = TfidfVectorizer()
train_cosine_matrix = cosine_tfidf.fit_transform(
    train_review_train_df['comment_clean']
)
test_cosine_matrix = cosine_tfidf.transform(
    train_review_test_df['comment_clean']
)

train_similarity_matrix = cosine_similarity(
    train_cosine_matrix,
    dense_output=False
)
train_similarity_matrix.setdiag(0)
train_similarity_matrix.eliminate_zeros()

train_review_train_df['max_cosine_similarity'] = (
    train_similarity_matrix.max(axis=1)
    .toarray()
    .ravel()
)

train_review_test_df['max_cosine_similarity'] = (
    cosine_similarity(
        test_cosine_matrix,
        train_cosine_matrix,
        dense_output=False
    )
    .max(axis=1)
    .toarray()
    .ravel()
)

model_tfidf = TfidfVectorizer()
X_train_text = model_tfidf.fit_transform(
    train_review_train_df['comment_clean']
)
X_test_text = model_tfidf.transform(
    train_review_test_df['comment_clean']
)

X_train = hstack([
    X_train_text,
    train_review_train_df[model_feature_cols]
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
    .values
], format='csr')

X_test = hstack([
    X_test_text,
    train_review_test_df[model_feature_cols]
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
    .values
], format='csr')

y_train = train_review_train_df['label'].astype(int)
y_test = train_review_test_df['label'].astype(int)

pd.DataFrame({
    'matrix': ['X_train', 'X_test'],
    'shape': [X_train.shape, X_test.shape]
})

,matrix,shape
0,X_train,"(68, 488)"
1,X_test,"(31, 488)"


In [26]:
# Train Logistic Regression Model

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)


lr_model.fit(
    X_train,
    y_train
)


lr_pred = lr_model.predict(X_test)

In [27]:
# Train Random Forest Classifier Model

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)


rf_model.fit(
    X_train,
    y_train
)


rf_pred = rf_model.predict(X_test)

In [28]:
# Evaluate Models

print("Logistic Regression")
print(classification_report(y_test, lr_pred))


print("Random Forest")
print(classification_report(y_test, rf_pred))

Logistic Regression
              precision    recall  f1-score   support

           0       0.71      1.00      0.83        15
           1       1.00      0.62      0.77        16

    accuracy                           0.81        31
   macro avg       0.86      0.81      0.80        31
weighted avg       0.86      0.81      0.80        31

Random Forest
              precision    recall  f1-score   support

           0       0.83      1.00      0.91        15
           1       1.00      0.81      0.90        16

    accuracy                           0.90        31
   macro avg       0.92      0.91      0.90        31
weighted avg       0.92      0.90      0.90        31



## 5. Preprocessing dan Feature Engineering untuk review_df

Bagian ini memakai alur preprocessing dan feature engineering yang sama untuk `review_df`, tanpa masuk ke tahap model.

In [29]:
# Data Cleaning dan Preprocessing untuk review_df

review_df = review_df.dropna(subset=["comment"]).copy()

review_df['comment_token'] = review_df['comment'].apply(
    lambda x: word_tokenize(str(x))
)

review_df['comment_lower'] = review_df['comment_token'].apply(
    lambda x: [word.lower() for word in x]
)

review_df['comment_punct'] = review_df['comment_lower'].apply(
    lambda x: [word for word in x if re.search(r'\w', word)]
)

review_stop_words = set(stopwords.words('indonesian'))

review_df['comment_stopword'] = review_df['comment_punct'].apply(
    lambda x: [word for word in x if word not in review_stop_words]
)

review_df['comment_clean'] = review_df['comment_punct'].apply(
    lambda x: ' '.join(x)
)

review_temp_cols = [
    'comment_token',
    'comment_lower',
    'comment_punct',
    'comment_stopword'
]

review_df = review_df.drop(
    columns=[col for col in review_temp_cols if col in review_df.columns]
)

review_df[['orderid', 'comment', 'comment_clean']].head()

,orderid,comment,comment_clean
1,NaN,work tp proses lama,work tp proses lama
3,NaN,"Respon cepat, work, dan proses agak lama",respon cepat work dan proses agak lama
5,NaN,Mantull work,mantull work
13,NaN,"Sangat memuaskan belanja di sini, akun original indonesia, harga murah, penjual respon cepet",sangat memuaskan belanja di sini akun original indonesia harga murah penjual respon cepet
15,NaN,"Josss lgs bs login netflix, terpercaya. Region indo netflixnya jadi aman",josss lgs bs login netflix terpercaya region indo netflixnya jadi aman


In [30]:
# Feature Engineering - TF-IDF untuk review_df

review_tfidf = TfidfVectorizer()

review_tfidf_matrix = review_tfidf.fit_transform(
    review_df['comment_clean']
)

review_tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 41533 stored elements and shape (3688, 4978)>

In [31]:
review_tfidf_preview_df = pd.DataFrame(
    review_tfidf_matrix[:5].toarray(),
    columns=review_tfidf.get_feature_names_out(),
    index=review_df.index[:5]
)

review_tfidf_preview_df

,00,000,01,10,100,1000,10000,100k,100x,103,10hari,10k,10m,10menit,10mnt,11,12,123,13,132961157,135,15,1500s,15mnt,15x,16,169k,17,17109733,19,1922,1bln,1bulan,1hari,1mggu,1minggu,1tahun,1th,1thn,1x,1x24,1x24jam,20,2013,2020,20rb,20rebu,23,24,24jam,...,yeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeeey,yeeeesss,yeeeeyyy,yeeey,yeeeyy,yeeyyyy,yes,yess,yesss,yey,yeyeye,yeyy,yeyyy,yeyyyy,yg,yggggrribjfjcupfhdkydiydkufouflufluxccylsiywiypiflhxmgxkgxkhxlhdhlxh,yogyakarta,yok,yokkkkkkk,yooo,yoooo,you,your,youu,youuu,youuuu,youuuuuu,youuuuuuu,youuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuu,ys,yuahh,yuhuu,yuhuuu,yuhuuuuu,yuhuuuuy,yuhuuuuybf,yuhuuuy,yuhuuuyyy,yuhuuw,yuhuuy,yuk,yusuf,yuuuhuuu,yuuuu,yuuuuuuuuuuuuuuuuuuuuuuu,yyaaaa,zonk,zonkk,zuper,الله
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [32]:
# Feature Engineering
# Review Length untuk review_df

review_df['review_length'] = review_df['comment_clean'].apply(
    lambda x: len(x.split())
)

review_df[['orderid', 'comment', 'comment_clean', 'review_length']].head()

,orderid,comment,comment_clean,review_length
1,NaN,work tp proses lama,work tp proses lama,4
3,NaN,"Respon cepat, work, dan proses agak lama",respon cepat work dan proses agak lama,7
5,NaN,Mantull work,mantull work,2
13,NaN,"Sangat memuaskan belanja di sini, akun original indonesia, harga murah, penjual respon cepet",sangat memuaskan belanja di sini akun original indonesia harga murah penjual respon cepet,13
15,NaN,"Josss lgs bs login netflix, terpercaya. Region indo netflixnya jadi aman",josss lgs bs login netflix terpercaya region indo netflixnya jadi aman,11


In [33]:
# Feature Engineering
# Positive Word Count and Ratio untuk review_df

review_positive_words = {
    'bagus', 'mantap', 'baik', 'recommended', 'rekomendasi',
    'terbaik', 'murah', 'cepat', 'puas', 'suka', 'keren',
    'oke', 'ok', 'aman', 'rapi', 'sesuai', 'worth', 'top'
}

review_df['positive_word_count'] = review_df['comment_clean'].apply(
    lambda x: sum(1 for word in x.split() if word in review_positive_words)
)

review_df['positive_word_ratio'] = review_df.apply(
    lambda row: row['positive_word_count'] / row['review_length']
    if row['review_length'] > 0 else 0,
    axis=1
)

review_df[
    [
        'comment',
        'comment_clean',
        'positive_word_count',
        'positive_word_ratio'
    ]
].head()

,comment,comment_clean,positive_word_count,positive_word_ratio
1,work tp proses lama,work tp proses lama,0,0.000000
3,"Respon cepat, work, dan proses agak lama",respon cepat work dan proses agak lama,1,0.142857
5,Mantull work,mantull work,0,0.000000
13,"Sangat memuaskan belanja di sini, akun original indonesia, harga murah, penjual respon cepet",sangat memuaskan belanja di sini akun original indonesia harga murah penjual respon cepet,1,0.076923
15,"Josss lgs bs login netflix, terpercaya. Region indo netflixnya jadi aman",josss lgs bs login netflix terpercaya region indo netflixnya jadi aman,1,0.090909


In [34]:
# Feature Engineering
# Repeated Word Count and Ratio untuk review_df

review_df['repeated_word_count'] = review_df['comment_clean'].apply(
    lambda x: sum(
        count - 1
        for count in Counter(x.split()).values()
        if count > 1
    )
)

review_df['repetition_score'] = review_df.apply(
    lambda row: row['repeated_word_count'] / row['review_length']
    if row['review_length'] > 0 else 0,
    axis=1
)

review_df[
    [
        'comment',
        'repeated_word_count',
        'repetition_score'
    ]
].head()

,comment,repeated_word_count,repetition_score
1,work tp proses lama,0,0.0
3,"Respon cepat, work, dan proses agak lama",0,0.0
5,Mantull work,0,0.0
13,"Sangat memuaskan belanja di sini, akun original indonesia, harga murah, penjual respon cepet",0,0.0
15,"Josss lgs bs login netflix, terpercaya. Region indo netflixnya jadi aman",0,0.0


In [35]:
# Feature Engineering
# Frequency of Reviews by User untuk review_df

review_df['ctime_datetime'] = pd.to_datetime(
    review_df['ctime'],
    unit='s',
    errors='coerce'
)

review_df['review_date'] = (
    review_df['ctime_datetime'].dt.date
)

review_df['user_review_count'] = (
    review_df.groupby('userid')['userid']
    .transform('count')
)

review_df['user_review_per_day'] = (
    review_df
    .groupby(['userid', 'review_date'])['userid']
    .transform('count')
)

review_df[
    [
        'userid',
        'comment',
        'user_review_count',
        'user_review_per_day'
    ]
].head()

,userid,comment,user_review_count,user_review_per_day
1,15569414.0,work tp proses lama,1,1
3,140324785.0,"Respon cepat, work, dan proses agak lama",1,1
5,129739160.0,Mantull work,1,1
13,267520353.0,"Sangat memuaskan belanja di sini, akun original indonesia, harga murah, penjual respon cepet",1,1
15,267589554.0,"Josss lgs bs login netflix, terpercaya. Region indo netflixnya jadi aman",1,1


In [36]:
# Feature Engineering
# Cosine Similarity of Reviews untuk review_df

review_similarity_matrix = cosine_similarity(
    review_tfidf_matrix,
    dense_output=False
)

review_similarity_matrix.setdiag(0)
review_similarity_matrix.eliminate_zeros()

review_df['max_cosine_similarity'] = (
    review_similarity_matrix.max(axis=1)
    .toarray()
    .ravel()
)

review_df[
    [
        'comment',
        'max_cosine_similarity'
    ]
].head()

,comment,max_cosine_similarity
1,work tp proses lama,0.641670
3,"Respon cepat, work, dan proses agak lama",0.641670
5,Mantull work,0.776947
13,"Sangat memuaskan belanja di sini, akun original indonesia, harga murah, penjual respon cepet",0.402735
15,"Josss lgs bs login netflix, terpercaya. Region indo netflixnya jadi aman",0.356952


## 6. Semi Supervised Modelling

RF = Random Forest, RL = Regresi Logistik. Data test dari split `train_review_df` tetap dipisah untuk evaluasi.

In [37]:
# Pakai split train-test dari train_review_df yang sudah dibuat di AI Modeling

train_texts = set(train_review_train_df['comment_clean'])
leaked_test = train_review_test_df[
    train_review_test_df['comment_clean'].isin(train_texts)
]

pd.DataFrame({
    'data': ['train_review_train_df', 'train_review_test_df', 'leaked_test'],
    'rows': [len(train_review_train_df), len(train_review_test_df), len(leaked_test)],
    'label_fake': [
        (train_review_train_df['fakeornot'] == 'fake').sum(),
        (train_review_test_df['fakeornot'] == 'fake').sum(),
        (leaked_test['fakeornot'] == 'fake').sum()
    ],
    'label_original': [
        (train_review_train_df['fakeornot'] == 'original').sum(),
        (train_review_test_df['fakeornot'] == 'original').sum(),
        (leaked_test['fakeornot'] == 'original').sum()
    ]
})

,data,rows,label_fake,label_original
0,train_review_train_df,68,34,34
1,train_review_test_df,31,16,15
2,leaked_test,0,0,0


In [38]:
# TF-IDF dan fitur numerik untuk train/test/review

num_cols = [
    'review_length',
    'repetition_score',
    'positive_word_ratio',
    'user_review_count',
    'user_review_per_day',
    'max_cosine_similarity'
]

cosine_tfidf = TfidfVectorizer()
train_cosine_matrix = cosine_tfidf.fit_transform(
    train_review_train_df['comment_clean']
)
test_cosine_matrix = cosine_tfidf.transform(
    train_review_test_df['comment_clean']
)
review_cosine_matrix = cosine_tfidf.transform(
    review_df['comment_clean']
)

train_similarity_matrix = cosine_similarity(
    train_cosine_matrix,
    dense_output=False
)
train_similarity_matrix.setdiag(0)
train_similarity_matrix.eliminate_zeros()

train_review_train_df['max_cosine_similarity'] = (
    train_similarity_matrix.max(axis=1)
    .toarray()
    .ravel()
)
train_review_test_df['max_cosine_similarity'] = (
    cosine_similarity(test_cosine_matrix, train_cosine_matrix, dense_output=False)
    .max(axis=1)
    .toarray()
    .ravel()
)
review_df['max_cosine_similarity'] = (
    cosine_similarity(review_cosine_matrix, train_cosine_matrix, dense_output=False)
    .max(axis=1)
    .toarray()
    .ravel()
)

semi_tfidf = TfidfVectorizer()
semi_tfidf.fit(
    pd.concat(
        [
            train_review_train_df['comment_clean'],
            review_df['comment_clean']
        ],
        ignore_index=True
    )
)

X_train_semi = hstack([
    semi_tfidf.transform(train_review_train_df['comment_clean']),
    train_review_train_df[num_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values
], format='csr')

X_test_semi = hstack([
    semi_tfidf.transform(train_review_test_df['comment_clean']),
    train_review_test_df[num_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values
], format='csr')

X_review_semi = hstack([
    semi_tfidf.transform(review_df['comment_clean']),
    review_df[num_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values
], format='csr')

y_train_semi = train_review_train_df['label'].astype(int)
y_test_semi = train_review_test_df['label'].astype(int)

pd.DataFrame({
    'matrix': ['X_train_semi', 'X_test_semi', 'X_review_semi'],
    'shape': [X_train_semi.shape, X_test_semi.shape, X_review_semi.shape]
})

,matrix,shape
0,X_train_semi,"(68, 4984)"
1,X_test_semi,"(31, 4984)"
2,X_review_semi,"(3688, 4984)"


In [39]:
# Train awal RF dan RL, lalu prediksi review_df

rf_model_semi = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
rf_model_semi.fit(X_train_semi, y_train_semi)

rl_model_semi = LogisticRegression(
    max_iter=1000,
    random_state=42
)
rl_model_semi.fit(X_train_semi, y_train_semi)

review_df['rf_pred'] = pd.Series(
    rf_model_semi.predict(X_review_semi),
    index=review_df.index
).map(label_map)

review_df['lr_pred'] = pd.Series(
    rl_model_semi.predict(X_review_semi),
    index=review_df.index
).map(label_map)

review_df['rf_confidence'] = rf_model_semi.predict_proba(X_review_semi).max(axis=1)
review_df['lr_confidence'] = rl_model_semi.predict_proba(X_review_semi).max(axis=1)

review_df['rl_semi_label'] = review_df['lr_pred']
review_df['rl_semi_confidence'] = review_df['lr_confidence']

review_df[
    [
        'rf_pred',
        'lr_pred',
        'rf_confidence',
        'lr_confidence',
        'comment'
    ]
].head()

,rf_pred,lr_pred,rf_confidence,lr_confidence,comment
1,original,original,0.50,0.700518,work tp proses lama
3,original,original,0.68,0.733134,"Respon cepat, work, dan proses agak lama"
5,fake,original,0.70,0.685491,Mantull work
13,original,original,0.70,0.787937,"Sangat memuaskan belanja di sini, akun original indonesia, harga murah, penjual respon cepet"
15,original,original,0.78,0.800731,"Josss lgs bs login netflix, terpercaya. Region indo netflixnya jadi aman"


In [40]:
# Ambil pseudo-label yang disetujui RF dan RL, lalu gabungkan dengan train split

pseudo_mask = (
    (review_df['lr_pred'] == review_df['rf_pred']) &
    (review_df['rf_confidence'] >= 0.90) &
    (review_df['lr_confidence'] >= 0.80)
)

pseudo_df = review_df[pseudo_mask].copy()
pseudo_df['fakeornot'] = pseudo_df['rl_semi_label']
pseudo_df['label'] = pseudo_df['fakeornot'].map({
    'original': 0,
    'fake': 1
})

semi_train_with_pseudo_df = pd.concat(
    [
        train_review_train_df,
        pseudo_df.reindex(columns=train_review_train_df.columns)
    ],
    ignore_index=True
)

X_train_with_pseudo = vstack([
    X_train_semi,
    X_review_semi[pseudo_mask.to_numpy()]
])

y_train_with_pseudo = pd.concat(
    [
        y_train_semi.reset_index(drop=True),
        pseudo_df['label'].reset_index(drop=True)
    ],
    ignore_index=True
).astype(int)

pd.DataFrame({
    'data': [
        'train_review_train_df',
        'train_review_test_df',
        'pseudo_df',
        'semi_train_with_pseudo_df'
    ],
    'rows': [
        len(train_review_train_df),
        len(train_review_test_df),
        len(pseudo_df),
        len(semi_train_with_pseudo_df)
    ]
})

,data,rows
0,train_review_train_df,68
1,train_review_test_df,31
2,pseudo_df,40
3,semi_train_with_pseudo_df,108


In [41]:
# Distribusi label pada pseudo_df dan data train gabungan

display(pseudo_df['fakeornot'].value_counts().rename('pseudo_label_count'))
display(semi_train_with_pseudo_df['fakeornot'].value_counts().rename('combined_label_count'))

fakeornot
fake        39
original     1
Name: pseudo_label_count, dtype: int64

fakeornot
fake        73
original    35
Name: combined_label_count, dtype: int64

In [42]:
# Train ulang RF dan RL dengan train split + pseudo_df, lalu evaluasi pada test split

rf_model_semi = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
rf_model_semi.fit(X_train_with_pseudo, y_train_with_pseudo)

rl_model_semi = LogisticRegression(
    max_iter=1000,
    random_state=42
)
rl_model_semi.fit(X_train_with_pseudo, y_train_with_pseudo)

rf_test_pred = rf_model_semi.predict(X_test_semi)
rl_test_pred = rl_model_semi.predict(X_test_semi)

print('RF Semi-Supervised')
print(classification_report(
    y_test_semi,
    rf_test_pred,
    labels=[0, 1],
    target_names=['original', 'fake']
))

print('RL Semi-Supervised')
print(classification_report(
    y_test_semi,
    rl_test_pred,
    labels=[0, 1],
    target_names=['original', 'fake']
))

RF Semi-Supervised
              precision    recall  f1-score   support

    original       0.82      0.93      0.88        15
        fake       0.93      0.81      0.87        16

    accuracy                           0.87        31
   macro avg       0.88      0.87      0.87        31
weighted avg       0.88      0.87      0.87        31

RL Semi-Supervised
              precision    recall  f1-score   support

    original       0.71      1.00      0.83        15
        fake       1.00      0.62      0.77        16

    accuracy                           0.81        31
   macro avg       0.86      0.81      0.80        31
weighted avg       0.86      0.81      0.80        31



In [43]:
# Confusion matrix RF dan RL

rf_confusion_df = pd.crosstab(
    y_test_semi.reset_index(drop=True).map(label_map),
    pd.Series(rf_test_pred).map(label_map),
    rownames=['Actual'],
    colnames=['RF Predicted']
).reindex(
    index=['original', 'fake'],
    columns=['original', 'fake'],
    fill_value=0
)

rl_confusion_df = pd.crosstab(
    y_test_semi.reset_index(drop=True).map(label_map),
    pd.Series(rl_test_pred).map(label_map),
    rownames=['Actual'],
    colnames=['RL Predicted']
).reindex(
    index=['original', 'fake'],
    columns=['original', 'fake'],
    fill_value=0
)

display(rf_confusion_df)
display(rl_confusion_df)

RF Predicted,original,fake
Actual,,
original,14,1
fake,3,13


RL Predicted,original,fake
Actual,,
original,15,0
fake,6,10


In [44]:
# Prediksi final pada review_df menggunakan model semi-supervised

review_df['rf_semi_label'] = pd.Series(
    rf_model_semi.predict(X_review_semi),
    index=review_df.index
).map(label_map)

review_df['rl_semi_label'] = pd.Series(
    rl_model_semi.predict(X_review_semi),
    index=review_df.index
).map(label_map)

review_df['rf_semi_confidence'] = rf_model_semi.predict_proba(X_review_semi).max(axis=1)
review_df['rl_semi_confidence'] = rl_model_semi.predict_proba(X_review_semi).max(axis=1)

review_df[
    [
        'rf_semi_label',
        'rl_semi_label',
        'rf_semi_confidence',
        'rl_semi_confidence',
        'comment'
    ]
].head()

,rf_semi_label,rl_semi_label,rf_semi_confidence,rl_semi_confidence,comment
1,original,original,0.52,0.703959,work tp proses lama
3,original,original,0.61,0.738296,"Respon cepat, work, dan proses agak lama"
5,fake,original,0.69,0.689928,Mantull work
13,original,original,0.61,0.793788,"Sangat memuaskan belanja di sini, akun original indonesia, harga murah, penjual respon cepet"
15,original,original,0.78,0.802874,"Josss lgs bs login netflix, terpercaya. Region indo netflixnya jadi aman"


In [45]:
# RF: 10 fake dan 10 original dari review_df

rf_fake_sample = (
    review_df[review_df['rf_semi_label'] == 'fake']
    .sort_values('rf_semi_confidence', ascending=False)
    .head(10)
)

rf_original_sample = (
    review_df[review_df['rf_semi_label'] == 'original']
    .sort_values('rf_semi_confidence', ascending=False)
    .head(10)
)

pd.concat(
    [rf_fake_sample, rf_original_sample],
    keys=['fake', 'original'],
    names=['sample_group']
)[
    [
        'rf_semi_label',
        'rf_semi_confidence',
        'comment',
        'product_title',
        'rating_star',
        'review_length',
        'positive_word_ratio',
        'repetition_score',
        'user_review_count',
        'user_review_per_day',
        'max_cosine_similarity'
    ]
]

rf_semi_label  rf_semi_confidence  \
sample_group                                          
fake         2263          fake                1.00   
             2265          fake                1.00   
             2267          fake                1.00   
             2269          fake                1.00   
             2271          fake                1.00   
             8795          fake                1.00   
             2301          fake                1.00   
             283           fake                1.00   
             281           fake                1.00   
             279           fake                1.00   
original     4951      original                0.97   
             3507      original                0.92   
             6227      original                0.91   
             131       original                0.90   
             7619      original                0.90   
             2899      original                0.89   
             413       original                0.89   
             2935      original                0.88   
             4767      original                0.88   
             1325      original                0.87   

                                                                                                                                   comment  \
sample_group                                                                                                                                 
fake         2263  Produk nya berkualitas banget\r\nSeller responsif ehehe\r\nFastresp juga termasuknya\r\nTrusted kok ehehe,udah banya...   
             2265  Produk nya berkualitas banget\r\nSeller responsif ehehe\r\nFastresp juga termasuknya\r\nTrusted kok ehehe,udah banya...   
             2267  Produk nya berkualitas banget\r\nSeller responsif ehehe\r\nFastresp juga termasuknya\r\nTrusted kok ehehe,udah banya...   
             2269  Produk nya berkualitas banget\r\nSeller responsif ehehe\r\nFastresp juga termasuknya\r\nTrusted kok ehehe,udah banya...   
             2271  Produk nya berkualitas banget\r\nSeller responsif ehehe\r\nFastresp juga termasuknya\r\nTrusted kok ehehe,udah banya...   
             8795  Seller selalu mantapp..\r\nUdah orederann yang ke belasan kalinya\r\nTidak pernah mengecewakann\r\nRecommend seller ...   
             2301  Produk nya berkualitas banget\r\nSeller responsif ehehe\r\nFastresp juga termasuknya\r\nTrusted kok ehehe,udah banya...   
             283                                                             Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy   
             281                                                             Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy   
             279                                                             Mantabbagusmantabbagus yuhuuy\r\nRecommended seller yuhuuuyyy   
original     4951  Penjual fast respon bgt. Padahal sudah dini hari tapi masih fast respon. Terima kasih aku jd bisa nonton di netflix ...   
             3507  Respon sellernya kurang fast respon, tapi sangat membantu bagi aku yg baru langganan netflix. Semoga akunnya gaada k...   
             6227  Subs nya Annual jadi aman, semoga aja ga kedeteksi jadi ga ada masalah\r\n\r\nSemoga garansi benar dibantu\r\n\r\nAP...   
             131   Nyoba yg 1 bulan dulu. Admin fast respon parah. Jam 11 malem pun diladenin haha. Proses cepat gasampe 5 menit uda di...   
             7619  Pemesanan kedua. Bulan pertama aman lancar tanpa ada kendala sama sekali. Seller fast respon dan baik. Pertahankan k...   
             2899  Akhirnya bisa nonton pake indihome hehe,tapi sayang di netflix indo belum ada riverdale s4,dikira netflixnya sama aj...   
             413   Selalu puas order di sini ini orderan kedua dan lancar terus tanpa kendala thank you buat admin nya yg selalu fast r...   
             2935  JANGAN BELI DISINI NJIR, PROSES LAMAAAAAAA, UDAH GITU AKUNNYA BARU 2 HARI DIPAKE KENA HOLD

In [46]:
# RL: 10 fake dan 10 original dari review_df

rl_fake_sample = (
    review_df[review_df['rl_semi_label'] == 'fake']
    .sort_values('rl_semi_confidence', ascending=False)
    .head(10)
)

rl_original_sample = (
    review_df[review_df['rl_semi_label'] == 'original']
    .sort_values('rl_semi_confidence', ascending=False)
    .head(10)
)

pd.concat(
    [rl_fake_sample, rl_original_sample],
    keys=['fake', 'original'],
    names=['sample_group']
)[
    [
        'rl_semi_label',
        'rl_semi_confidence',
        'comment',
        'product_title',
        'rating_star',
        'review_length',
        'positive_word_ratio',
        'repetition_score',
        'user_review_count',
        'user_review_per_day',
        'max_cosine_similarity'
    ]
]

rl_semi_label  rl_semi_confidence  \
sample_group                                          
fake         67            fake            1.000000   
             69            fake            1.000000   
             95            fake            1.000000   
             2495          fake            1.000000   
             2295          fake            1.000000   
             2297          fake            1.000000   
             2299          fake            1.000000   
             2301          fake            1.000000   
             2305          fake            1.000000   
             2307          fake            1.000000   
original     1577      original            0.978049   
             333       original            0.972301   
             7307      original            0.971344   
             5489      original            0.969196   
             2609      original            0.968760   
             1217      original            0.967272   
             5805      original            0.965519   
             2929      original            0.965029   
             1625      original            0.964656   
             41        original            0.963953   

                                                                                                                                   comment  \
sample_group                                                                                                                                 
fake         67                                                  Oke bgt pokoknya keren pengiriman cepat dan harganya cocok untuk reseller   
             69                                                  Oke bgt pokoknya keren pengiriman cepat dan harganya cocok untuk reseller   
             95                 Hp sampai dengan selamat, berfungsi dengan baik, tiada kendala Alhamdulillah, semoga awet dan mantap jiwaa   
             2495                                                                SUKA MANTAPPU JIWAAA YUHUUU MANTEPPPPPPPPPPPPPPPPPPPPPPPP   
             2295  Produk nya berkualitas banget\r\nSeller responsif ehehe\r\nFastresp juga termasuknya\r\nTrusted kok ehehe,udah banya...   
             2297  Produk nya berkualitas banget\r\nSeller responsif ehehe\r\nFastresp juga termasuknya\r\nTrusted kok ehehe,udah banya...   
             2299  Produk nya berkualitas banget\r\nSeller responsif ehehe\r\nFastresp juga termasuknya\r\nTrusted kok ehehe,udah banya...   
             2301  Produk nya berkualitas banget\r\nSeller responsif ehehe\r\nFastresp juga termasuknya\r\nTrusted kok ehehe,udah banya...   
             2305                                                     Recommended seller\r\nMantap jiwaaa\r\nResponsif dan pelayanan cepat   
             2307                                                     Recommended seller\r\nMantap jiwaaa\r\nResponsif dan pelayanan cepat   
original     1577  Proses agk lama dpt akun, tp gk smpai sehari. Dan sdh bs dipakai. Smg gk kena on hold atw prubhan passwrd. Krena cm ...   
             333   Sip mantap walpun slowres tp emg mantap si terbaek uda 2x beli\r\nRevisi! Ngilang seller nya ngilang punya gw incorr...   
             7307  Mantul (Mantap Betul) recomend, tapi masih belum tau apa betul sampe 1 bln, boleh nanti aku hitung kan kmrn beli 3 h...   
             5489  Cocok banget beli di sini awalnya. Tapi ternyata akunnya akun luar negri. Jadi pas netflix ada perubhan sistem . Rib...   
             2609  Sedikit agak susah untuk smart TV...VPN nya..termasuk PC dan Laptop VPN nya..terbatas jd ya gak bisa nonton Film 30 ...   
             1217  Sblm order tanya' dulu. Dan semua di jawab dgn jelas. Semua pertanyaan sy di ladenin sama admin. Mantap banger admin...   
             5805  It works! Terima kasih min! Mungkin kalo ada yg gabisa download apk dari apkpure atau site lain, bisa dicoba untuk g...   
             2929  Akun netflixnya sejauh ini ga bermasalah, cuma nord vpn nya yg bermasalah gatau kenapa net